In [1]:
%matplotlib inline

import sys
sys.path.append('..')

import optimus
import numpy as np

barycentre_pressure_list = []

frequencies = np.logspace(1.30103, 4.34341, num = 500)

for frequency in frequencies:
    # frequency = 2500
    source = optimus.source.create_planewave(frequency, direction=(-1,0,0))

    optimus.global_parameters.incident_field_parallelisation.parallelisation_method = "multiprocessing"

    material_ext = optimus.material.load_material('air')

    material_user_1 = optimus.material.create_material(name='abdomen',
                                                   density=917.0,
                                                   speed_of_sound=1412.0,
                                                   attenuation_coeff_a=595.3,
                                                   attenuation_pow_b=1)
    material_user_2 = optimus.material.create_material(name='bone',
                                                   density=1912.0,
                                                   speed_of_sound=4080.0,
                                                   attenuation_coeff_a=23.1,
                                                   attenuation_pow_b=1)
    material_user_3 = optimus.material.create_material(name='amniotic_fluid',
                                                   density=1000.0,
                                                   speed_of_sound=1500.0,
                                                   attenuation_coeff_a=15e-3,
                                                   attenuation_pow_b=2)

    materials_int = [material_user_1, material_user_2, material_user_3]

    wavelengths = []
    for material in [material_ext] + materials_int:
        wavelengths.append(material.compute_wavelength(frequency))
        print("wavelength in " + material.name + " is: ", wavelengths[-1]*1e3, 'mm')

    
    # changed to include change meshes depending on frequency 
    if frequency < 115.2: 
        file1 = 'Data/new_meshes/ABDO_20.msh'
        file2 = 'Data/new_meshes/SPINE_6.67.msh'
        file3 = 'Data/new_meshes/UTERUS_10.msh'
    elif frequency >= 115.2 and frequency < 663.6:
        file1 = 'Data/new_meshes/ABDO_14.38.msh'
        file2 = 'Data/new_meshes/SPINE_4.78.msh'
        file3 = 'Data/new_meshes/UTERUS_7.34.msh'
    elif frequency >= 663.6 and frequency < 3822.1:
        file1 = 'Data/new_meshes/ABDO_8.76.msh'
        file2 = 'Data/new_meshes/SPINE_2.89.msh'
        file3 = 'Data/new_meshes/UTERUS_4.67.msh'
    elif frequency >= 3822.1:
        file1 = 'Data/new_meshes/ABDO_3.14.msh'
        file2 = 'Data/new_meshes/SPINE_1.msh'
        file3 = 'Data/new_meshes/UTERUS_2.msh'

    geometry1 = optimus.geometry.load.import_grid(file1, label="abdomen")
    geometry1.scale_grid(1e-3)
    geometry2 = optimus.geometry.load.import_grid(file2, label="spine")
    geometry2.scale_grid(1e-3)
    geometry3 = optimus.geometry.load.import_grid(file3, label="uterus")
    geometry3.scale_grid(1e-3)
    geometries = (geometry1, geometry2, geometry3)

    for geometry in geometries:
        print("number of vertices on " + geometry.label + " is: ", geometry.number_of_vertices())

    mygraph = optimus.geometry.Graph("Sonic Womb GS370")

    mygraph.create_exterior_domain(material_ext, source)

    mygraph.create_interface(geometry1, 0, material_user_1, verbose=False)
    mygraph.create_interface(geometry2, 1, material_user_2, verbose=False)
    mygraph.create_interface(geometry3, 1, material_user_3, verbose=False)


    model = optimus.model.create_nested_model(mygraph, frequency, "pmchwt", ["osrc","osrc","osrc"])

    postprocess_plane = optimus.postprocess.VisualisePlane(model)

    import numpy as np
    z_offset = np.mean(geometries[2].grid.leaf_view.vertices, axis=1)[2]


    # %%time
    model.solve()

    grid = geometries[2].grid
    vertices = grid.leaf_view.vertices
 
    import numpy as np 
    uterus_barycentre = np.mean(vertices, axis=1)

    uterus_barycentre.shape
    vertices.shape

    postprocess_cloud = optimus.postprocess.VisualiseCloudPoints(model)
    postprocess_cloud.create_computational_grid(points=uterus_barycentre)
    postprocess_cloud.compute_fields()

    barycentre_pressure = postprocess_cloud.field.total_field[0]
    
    barycentre_pressure_list.append(barycentre_pressure)


In [3]:
barycentre_pressure_array = np.array(barycentre_pressure_list)
print (barycentre_pressure_array) # shows all the pressures calculated at each frequency, going up in 100Hz
frequencies = np.array(list(range(100,2501,100)))
import scipy.io as sio
sio.savemat('sound_transmission.mat',{'barycentre_pressures': barycentre_pressure_list, 'frequencies':frequencies})

[ 0.88562836-0.38724398j  0.58174868-0.70516654j  0.18026776-0.88313203j
 -0.21747446-0.87398719j -0.52838127-0.6825633j  -0.69460726-0.36834991j
 -0.69228131-0.01953418j -0.53967935+0.27450789j -0.2913506 +0.44580967j
 -0.02154213+0.46461223j  0.19693845+0.34558819j  0.31124969+0.14119008j
  0.30136124-0.07671087j  0.18401527-0.23925247j  0.00632483-0.30046203j
 -0.16910592-0.24881277j -0.28408737-0.1084575j  -0.30220601+0.07002724j
 -0.21882304+0.22635731j -0.06219477+0.30953649j  0.11608917+0.29292419j
  0.25818288+0.18146238j  0.31802794+0.01053619j  0.27564703-0.16620716j
  0.14301533-0.29296599j]
